In [1]:
# Imports
import pandas as pd
import duckdb
import os
import sys

Download the following files from from https://zenodo.org/records/20247946 :
- SpotlightWeightSource_0102_0505_fullmatch_corrected.csv 
- entities_filtered_by_languages.csv

You also need: cross-verified-database.csv.gz

## Get sources linking to Napoleon

In [7]:
# Get data with old Spotlight predictions
fullmatch_df = pd.read_csv('data/SpotlightWeightSource_0102_0505_fullmatch_corrected.csv')
fullmatch_df.head()

,language_code,source_wikidata_id,target_wikidata_id,weight
0,en,Q6882,Q53003,0.0
1,en,Q6882,Q5912,0.0
2,en,Q6882,Q212719,0.0
3,en,Q6882,Q514998,0.0
4,en,Q6882,Q210134,0.0


In [3]:
# Only keep sources with Napoleon (Q517) as target and weight over 0
filtered_df = fullmatch_df[(fullmatch_df["target_wikidata_id"]=="Q517") & (fullmatch_df["weight"]>0.0) & (fullmatch_df["language_code"]=="en")]
filtered_df.shape

(1060, 4)

In [5]:
# Split into low / mid / high quantiles by weight rank
filtered_df['quantile'] = pd.qcut(filtered_df['weight'].rank(method='first'), q=3, labels=['low', 'mid', 'high'])

# Sample evenly - 45 total
groups = {'low': 15, 'mid': 15, 'high': 15}
napo_sample = pd.concat([
    filtered_df[filtered_df['quantile'] == group].sample(n=size, random_state=24)
    for group, size in groups.items()
])

print(napo_sample.groupby('quantile')['weight'].agg(['min', 'max', 'count']))

          min   max  count
quantile                  
low       1.0   1.0     15
mid       1.0   3.0     15
high      3.0  23.0     15


/var/folders/d1/9j4b18194g70dz3d5kkx7vqw0000gn/T/ipykernel_47074/1343385019.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['quantile'] = pd.qcut(filtered_df['weight'].rank(method='first'), q=3, labels=['low', 'mid', 'high'])
/var/folders/d1/9j4b18194g70dz3d5kkx7vqw0000gn/T/ipykernel_47074/1343385019.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(napo_sample.groupby('quantile')['weight'].agg(['min', 'max', 'count']))


## Build graph with sources linking to Napoleon

In [6]:
nap3 = filtered_df[filtered_df['source_wikidata_id']=="Q7721"]

In [7]:
napo_sample = pd.concat([napo_sample, nap3])

In [8]:
# List of napoleon sources
napo_sources_ids = napo_sample["source_wikidata_id"].values.tolist()

In [9]:
entities = pd.read_csv('data/entities_filtered_by_languages.csv')

# Only keep people we want
napo_sources = entities[entities['wikidata_id'].isin(napo_sources_ids)]

napo_sources.to_csv('data/napo_sources.csv', index=False)
napo_sources.head()

,wikidata_id,birth,bigperiod_birth,un_subregion,gender
519,Q1048,-100.0,1.Ancient History Before 500AD,Southern Europe,Male
2383,Q115995,1795.0,4.Mid Modern Period 1751-1900AD,Western Europe,Male
4437,Q12718,1798.0,4.Mid Modern Period 1751-1900AD,Western Europe,Male
5211,Q130221,1194.0,2.Post-Classical History 501-1500AD,Southern Europe,Male
6346,Q1342645,1739.0,3.Early Modern Period 1501-1750AD,Western Europe,Male


Run the following command in terminal:

```
python src/main.py build \
    --input data/napo_sources.csv \
    --output data/out/napo_sources.db \
    --languages en
```

## Write text files for manual annotation

In [ ]:
con = duckdb.connect('data/out/napo_sources.db')

rows = con.execute("""
    SELECT wp.language_code, wp.page_title, e.extract
    FROM wiki_page wp
    JOIN extract e ON e.wiki_page_id = wp.wiki_page_id
""").fetchall()

con.close()

output_dirs = [
    "data/text_files",
    "../hpc_jobs/hpc_relik/text_files",
    "../hpc_jobs/hpc_relik_large/text_files",
]

for language_code, page_title, extract in rows:
    filename = f"{page_title.replace(' ', '_')}_{language_code}.txt"
    for d in output_dirs:
        with open(os.path.join(d, filename), "w", encoding="utf-8") as f:
            f.write(extract or "")
    print(f"Wrote {filename}")

Wrote Friedrich_Wilhelm_Freiherr_von_Bülow_en.txt
Wrote Paolo_Virzì_en.txt
Wrote Gaspard_de_Prony_en.txt
Wrote Félicité_de_La_Mennais_en.txt
Wrote Achille_Baraguey_d'Hilliers_en.txt
Wrote Vincent-Marie_Viénot,_Count_of_Vaublanc_en.txt
Wrote Leo_Tolstoy_en.txt
Wrote Franz_Joseph_Gall_en.txt
Wrote Ida_Laura_Pfeiffer_en.txt
Wrote Napoleon_III_en.txt
Wrote Józef_Bem_en.txt
Wrote Archduke_Albrecht,_Duke_of_Teschen_en.txt
Wrote Ferdinand_Foch_en.txt
Wrote Donald_Adamson_en.txt
Wrote François_Christophe_de_Kellermann_en.txt
Wrote Maurice_d'Elbée_en.txt
Wrote Jacques_Laffitte_en.txt
Wrote Antonio_Canova_en.txt
Wrote Oscar_I_of_Sweden_en.txt
Wrote Robert_Clive_en.txt
Wrote Jean_Maximilien_Lamarque_en.txt
Wrote Daniel_O'Connell_en.txt
Wrote Georges_Cadoudal_en.txt
Wrote Mikhail_Tukhachevsky_en.txt
Wrote Prince_Philippe,_Count_of_Paris_en.txt
Wrote Germinal_Pierre_Dandelin_en.txt
Wrote Joseph_Haydn_en.txt
Wrote Frederick_II,_Holy_Roman_Emperor_en.txt
Wrote Prince_Josias_of_Saxe-Coburg-Saalfeld_en

## Get Spotlight weights

In [4]:
con = duckdb.connect('data/out/napo_sources.db')

source_biographies = con.execute("""
    SELECT 'Q' || wp.wikidata_id, wp.language_code, e.extract
    FROM wiki_page wp
    JOIN extract e ON e.wiki_page_id = wp.wiki_page_id
""").fetchall()

con.close()

source_biographies = pd.DataFrame(source_biographies, columns=["wikidata_id", "language_code", "extract"])
source_biographies.shape

(46, 3)

In [17]:
napoleon_links = napo_sample[["source_wikidata_id", "target_wikidata_id", "language_code"]]
napoleon_links.shape

(46, 3)

In [18]:
source_biographies.to_csv('../hpc_jobs/hpc_spotlight/data/source_biographies.csv', index=False)

napoleon_links.to_csv('../hpc_jobs/hpc_spotlight/data/napoleon_links.csv', index=False)

In [4]:
con = duckdb.connect('data/out/napo_sources.db')
source_names = con.execute("""
    SELECT 'Q' || wikidata_id AS wikidata_id, page_title
    FROM wiki_page WHERE language_code = 'en'
""").df()

con.close()

In [ ]:
con = duckdb.connect('data/out/napoleon_self.db')
napoleon_name = con.execute("""
    SELECT 'Q' || wikidata_id AS wikidata_id, page_title
    FROM wiki_page WHERE language_code = 'en'
""").df()

con.close()

In [6]:
source_names

,wikidata_id,page_title
0,Q63023,Friedrich Wilhelm Freiherr von Bülow
1,Q607190,Paolo Virzì
2,Q451608,Gaspard de Prony
3,Q334973,Félicité de La Mennais
4,Q115995,Achille Baraguey d'Hilliers
5,Q360481,"Vincent-Marie Viénot, Count of Vaublanc"
6,Q7243,Leo Tolstoy
7,Q57511,Franz Joseph Gall
8,Q78873,Ida Laura Pfeiffer
9,Q7721,Napoleon III


In [7]:
napoleon_name

,wikidata_id,page_title
0,Q517,Napoleon


In [8]:
all_names = pd.concat(
    [napoleon_name, source_names],
    ignore_index=True
)
all_names

,wikidata_id,page_title
0,Q517,Napoleon
1,Q63023,Friedrich Wilhelm Freiherr von Bülow
2,Q607190,Paolo Virzì
3,Q451608,Gaspard de Prony
4,Q334973,Félicité de La Mennais
5,Q115995,Achille Baraguey d'Hilliers
6,Q360481,"Vincent-Marie Viénot, Count of Vaublanc"
7,Q7243,Leo Tolstoy
8,Q57511,Franz Joseph Gall
9,Q78873,Ida Laura Pfeiffer


In [10]:
all_names.to_csv('../hpc_jobs/hpc_spotlight/data/napoleon_source_names.csv', index=False)

# Run spotlight job on HPC

NOT RELEVANT NOW

Open docker desktop.

Navigate to the `container` folder and run:
```
docker compose -f spotlight-compose.yml up spotlight.en
```

In [ ]:
sys.path.append(os.path.abspath("../src"))

from modules.spotlight_weight_source import SpotlightWeightSource

source = SpotlightWeightSource()
weights_df = source.assign_weights(
    entities_extracts_missing=source_biographies,
    links_df=napoleon_links,
    title_to_id_map={"en": {"Napoleon": "Q517"}},
)

weights_df.head()

weights_df.to_csv('../data/spotlight_weights.csv', index=False)

# Prep for Napoelon outgoing edges

In [10]:
con = duckdb.connect('data/out/napoleon_self.db')

napoleon_biography = con.execute("""
    SELECT 'Q' || wp.wikidata_id, wp.language_code, e.extract
    FROM wiki_page wp
    JOIN extract e ON e.wiki_page_id = wp.wiki_page_id
""").fetchall()

con.close()

napoleon_biography = pd.DataFrame(napoleon_biography, columns=["wikidata_id", "language_code", "extract"])

print(napoleon_biography.shape)
napoleon_biography.head()

(1, 3)


,wikidata_id,language_code,extract
0,Q517,en,Napoleon Bonaparte (born Napoleone di Buonapar...


In [9]:
# Edges from Napoleon (Q517) to other people
napo_out = fullmatch_df[
    (fullmatch_df["source_wikidata_id"] == "Q517")
    & (fullmatch_df["language_code"] == "en")
]

napoleon_out_links = napo_out[["source_wikidata_id", "target_wikidata_id", "language_code"]]

print(napoleon_out_links.shape)
napoleon_out_links.head()

(403, 3)


,source_wikidata_id,target_wikidata_id,language_code
103760,Q517,Q708011,en
103761,Q517,Q8442,en
103762,Q517,Q152463,en
103763,Q517,Q157099,en
103764,Q517,Q208086,en


In [11]:
napoleon_biography.to_csv('../hpc_jobs/hpc_spotlight/data/napoleon_out/napoleon_biography.csv', index=False)

napoleon_out_links.to_csv('../hpc_jobs/hpc_spotlight/data/napoleon_out/napoleon_out_links.csv', index=False)

In [ ]:
con = duckdb.connect('data/out/napo_targets.db')
target_names = con.execute("""
    SELECT 'Q' || wikidata_id AS wikidata_id, page_title
    FROM wiki_page WHERE language_code = 'en'
""").df()
con.close()

# assign_weights expects {lang: {page_title: wikidata_id}}
title_to_id_map = {
    "en": dict(zip(target_names["page_title"], target_names["wikidata_id"]))
}

403 titles, 0 targets missing


In [13]:
target_names

,wikidata_id,page_title
0,Q617241,François-Nicolas Vincent
1,Q242609,André Masséna
2,Q157521,Charles Lucien Bonaparte
3,Q151087,Jérôme Bonaparte
4,Q473506,"Louis Joseph, Dauphin of France"
...,...,...
398,Q152148,Wenceslaus IV of Bohemia
399,Q483405,Anacharsis Cloots
400,Q8404,John I of France
401,Q212068,Jean-Jacques Dessalines
